In [0]:
initial_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/bronze/day16_customer_files/day16_customers_initial.csv")
)
display(initial_df)
#initial_df.printSchema()

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z


In [0]:
initial_df.write.mode("overwrite").format("delta").saveAsTable("bronze.day16_customers")

In [0]:
%sql
SELECT *
FROM bronze.day16_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z


In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.day16_customers_scd1 (
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP
)
USING DELTA;

In [0]:
bronze_df = spark.table("bronze.day16_customers")
bronze_df.write.mode("overwrite").saveAsTable("silver.day16_customers_scd1")

In [0]:
%sql
SELECT *
FROM silver.day16_customers_scd1
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z


In [0]:
updates_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/bronze/day16_customer_files/day16_customers_updates.csv")
)
display(updates_df)

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Coimbatore,31,2026-08-24T09:00:00.000Z
102,Kumar,Bangalore,36,2026-08-24T09:05:00.000Z
104,Meena,Madurai,29,2026-08-24T09:10:00.000Z


In [0]:
from delta.tables import DeltaTable

silver_scd1 = DeltaTable.forName(
    spark,
    "silver.day16_customers_scd1"
)

(
    silver_scd1.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.CustomerId = source.CustomerId"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
SELECT *
FROM silver.day16_customers_scd1
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Coimbatore,31,2026-08-24T09:00:00.000Z
102,Kumar,Bangalore,36,2026-08-24T09:05:00.000Z
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z
104,Meena,Madurai,29,2026-08-24T09:10:00.000Z


In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.day16_customers_scd2 (
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP,
    StartDate TIMESTAMP,
    EndDate TIMESTAMP,
    IsCurrent BOOLEAN
)
USING DELTA;

In [0]:
from pyspark.sql.functions import col, lit

initial_scd2_df = (
    initial_df
    .withColumn("StartDate", col("UpdatedAt"))
    .withColumn("EndDate", lit(None).cast("timestamp"))
    .withColumn("IsCurrent", lit(True))
)
display(initial_scd2_df)
initial_scd2_df.write.mode("overwrite").saveAsTable("silver.day16_customers_scd2")

CustomerId,CustomerName,City,Age,UpdatedAt,StartDate,EndDate,IsCurrent
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z,2026-08-01T09:00:00.000Z,null,true
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z,2026-08-01T09:05:00.000Z,null,true
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z,2026-08-01T09:10:00.000Z,null,true


In [0]:
%sql
select * from silver.day16_customers_scd2;

CustomerId,CustomerName,City,Age,UpdatedAt,StartDate,EndDate,IsCurrent
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z,2026-08-01T09:00:00.000Z,null,true
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z,2026-08-01T09:05:00.000Z,null,true
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z,2026-08-01T09:10:00.000Z,null,true


In [0]:
updates_scd2_df = (
    updates_df
    .withColumn("StartDate", col("UpdatedAt"))
    .withColumn("EndDate", lit(None).cast("timestamp"))
    .withColumn("IsCurrent", lit(True))
)

In [0]:
current_df = (
    spark.table("silver.day16_customers_scd2")
    .filter(col("IsCurrent") == True)
)

changed_df = (
    updates_df.alias("source")
    .join(
        current_df.alias("target"),
        "CustomerId"
    )
    .filter(
        (col("source.CustomerName") != col("target.CustomerName")) |
        (col("source.City") != col("target.City")) |
        (col("source.Age") != col("target.Age"))
    )
)
display(changed_df)

CustomerId,CustomerName,City,Age,UpdatedAt,CustomerName,City,Age,UpdatedAt,StartDate,EndDate,IsCurrent
101,Arun,Coimbatore,31,2026-08-24T09:00:00.000Z,Arun,Chennai,30,2026-08-01T09:00:00.000Z,2026-08-01T09:00:00.000Z,null,true
102,Kumar,Bangalore,36,2026-08-24T09:05:00.000Z,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z,2026-08-01T09:05:00.000Z,null,true


In [0]:
silver_scd2 = DeltaTable.forName(
    spark,
    "silver.day16_customers_scd2"
)
(
    silver_scd2.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.CustomerId = source.CustomerId AND target.IsCurrent = true"
    )
    .whenMatchedUpdate(
        condition="""
            target.CustomerName <> source.CustomerName
            OR target.City <> source.City
            OR target.Age <> source.Age
        """,
        set={
            "EndDate": "source.UpdatedAt",
            "IsCurrent": "false"
        }
    )
    
    ).execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from silver.day16_customers_scd2;

CustomerId,CustomerName,City,Age,UpdatedAt,StartDate,EndDate,IsCurrent
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z,2026-08-01T09:10:00.000Z,null,true
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z,2026-08-01T09:00:00.000Z,2026-08-24T09:00:00.000Z,false
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z,2026-08-01T09:05:00.000Z,2026-08-24T09:05:00.000Z,false


In [0]:
new_versions_df = (
    updates_df
    .withColumn("StartDate", col("UpdatedAt"))
    .withColumn("EndDate", lit(None).cast("timestamp"))
    .withColumn("IsCurrent", lit(True))
)
new_versions_df.write.mode("append").saveAsTable(
    "silver.day16_customers_scd2"
)

In [0]:
%sql
SELECT
    CustomerId,
    CustomerName,
    City,
    Age,
    StartDate,
    EndDate,
    IsCurrent
FROM silver.day16_customers_scd2
ORDER BY CustomerId, StartDate;

CustomerId,CustomerName,City,Age,StartDate,EndDate,IsCurrent
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z,2026-08-24T09:00:00.000Z,false
101,Arun,Coimbatore,31,2026-08-24T09:00:00.000Z,null,true
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z,2026-08-24T09:05:00.000Z,false
102,Kumar,Bangalore,36,2026-08-24T09:05:00.000Z,null,true
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z,null,true
104,Meena,Madurai,29,2026-08-24T09:10:00.000Z,null,true
